<img width="20%" alt="EarthDaily Analytics" src="https://raw.githubusercontent.com/earthdaily/Images/main/Corporate/EarthDaily.png" style="border-radius: 15%">

# EarthDaily Agriculture - Visualization Engine Demo

Showcase of `earthdaily.agriculture.reporting.viz_engine` capabilities using real extraction data from the Coverage extractor.

## Step 1: Initialisation

In [ ]:
# Bootstrap: ensure src/ is on sys.path for earthdaily.agriculture imports
import sys
from pathlib import Path

_src = str(Path().resolve().parent / "src")
if _src not in sys.path:
    sys.path.insert(0, _src)

from earthdaily.agriculture.notebook_setup import init
init()

In [ ]:
1from earthdaily.agriculture.services.workflow_manager import WorkflowManager
manager = WorkflowManager("prod", log_to_console=True, log_level="WARNING")

## Step 2: Load entities

In [ ]:
manager.load_seasonfields()
entities = manager.sfd_list
print(f"Loaded {len(entities)} entities")
display(entities[["id", "name"]].head())

## Step 3: Run Coverage Extraction

Extract satellite coverage data to get a multi-type dataset:
- **float**: `coverage_percent` (0-100)
- **int**: `spatial_resolution` (10, 30, etc.)
- **categorical**: `sensor` (SENTINEL_2, LANDSAT_8, ...), `mask` (auto, native, ...)
- **date**: `date` (image acquisition date)
- **string**: `image_id`

In [ ]:
from earthdaily.agriculture.extractors.coverage_function import CoverageExtractor

cov_extractor = CoverageExtractor(
    manager.bearer_token, manager.token_expiration, config=manager.config
)

cov_extractor.setup_coverage_parameters(
    vegetation_index="NDVI",
    start_date="2024-01-01",
    end_date="2025-01-01",
    clear_cover_min=50,
    mask="auto",
)
# Extract coverage on 50 entities please change it in entities.head(50) if you want more or less
cov_result = cov_extractor.process_entity_coverage_bulk_parallel(
    entity_list=entities.head(50),
    max_workers=10,
    skip_export=True,
    prefix="viz_demo_coverage",
)

cov_df = cov_result["results_df"]
print(f"Coverage results: {len(cov_df)} rows, {cov_df['id'].nunique()} entities")
print(f"Columns: {list(cov_df.columns)}")
print(f"\nColumn types:")
for col in ["coverage_percent", "spatial_resolution", "sensor", "mask", "date"]:
    if col in cov_df.columns:
        print(f"  {col}: {cov_df[col].dtype} \u2014 {cov_df[col].nunique()} unique values")

In [ ]:
display(cov_df.head(10))

## Step 4: Define Column Specs

The viz_engine is driven by column specifications \u2014 a list of dicts describing
each column's type, display label, and optional aggregation/color rules.

Column types: `numeric`, `date`, `categorical`, `timeseries`

In [ ]:
# Column specs for coverage data
col_specs = [
    {
        "name": "coverage_percent",
        "label": "Coverage (%)",
        "type": "numeric",
        "agg": "mean",
        "color_scale": [
            {"above": 90, "color": "#4CAF50"},
            {"above": 70, "color": "#FFC107"},
            {"above": 0,  "color": "#F44336"},
        ],
    },
    {
        "name": "sensor",
        "label": "Sensor",
        "type": "categorical",
    },
    {
        "name": "mask",
        "label": "Cloud Mask",
        "type": "categorical",
    },
    {
        "name": "date",
        "label": "Image Date",
        "type": "date",
    },
]

# Also define a timeseries spec for coverage over time
ts_col_specs = [
    {
        "name": "coverage_percent",
        "label": "Coverage (%)",
        "type": "timeseries",
    },
]

print("Column specs defined:")
for s in col_specs:
    print(f"  {s['name']:25s} -> {s['type']}")

## Step 4b: Auto-Generate Viz Config

Instead of writing `col_specs` manually, use `generate_viz_config()` to auto-detect
column types from the DataFrame. It returns a config dict ready for all viz_engine functions,
and can optionally save it as YAML for later reuse.

**Workflow:**
1. `generate_viz_config(df)` -- inspects the DataFrame, classifies columns
2. Tune the generated YAML (rename labels, change types, add color scales)
3. `load_viz_config("path.yaml")` -- reload the tuned config

### 4b.1 Generate config from DataFrame

In [ ]:
from earthdaily.agriculture.reporting.viz_engine import generate_viz_config

# Auto-detect column types from the extraction results
auto_config = generate_viz_config(cov_df, entity_col="id", date_col="date")

In [ ]:
# Inspect the generated col_specs
for spec in auto_config["col_specs"]:
    print(f"  {spec['name']:35s} -> {spec['type']:12s}  ({spec['label']})")

### 4b.2 Save config as YAML for tuning

In [ ]:
# Save to YAML — edit the file to rename labels, change types, remove columns
generate_viz_config(
    cov_df,
    entity_col="id",
    date_col="date",
    save_to="results/coverage_demo_viz.yaml",
)

In [ ]:
# Preview the generated YAML
print(open("results/coverage_demo_viz.yaml").read())

### 4b.3 Load tuned config and use it

After editing the YAML (rename labels, remove unwanted columns, change `timeseries` to `numeric`, etc.),
reload it and use it with any viz_engine function.

In [ ]:
from earthdaily.agriculture.reporting.viz_engine import load_viz_config

# Load the saved (or tuned) config
loaded_config = load_viz_config("results/coverage_demo_viz.yaml")
from earthdaily.agriculture.reporting.viz_engine import timeseries_chart

# Use it directly with viz_engine functions
timeseries_chart(
    cov_df,
    loaded_config["col_specs"],
    entity_col=loaded_config["entity_col"],
    date_col=loaded_config["date_col"],
    mode="all",
)

In [ ]:
# Also works with distribution, KPI, per-entity charts
from earthdaily.agriculture.reporting.viz_engine import distribution_chart

distribution_chart(
    cov_df,
    loaded_config["col_specs"],
    entity_col=loaded_config["entity_col"],
    date_col=loaded_config["date_col"],
)

### 4b.4 Filter the auto-config for specific chart types

The generated config includes all columns. Filter `col_specs` to only the types
you need for a specific chart.

In [ ]:
# Filter to only timeseries columns for time series charts
ts_only = [s for s in loaded_config["col_specs"] if s["type"] == "timeseries"]
print(f"Timeseries columns: {[s['name'] for s in ts_only]}")

timeseries_chart(
    cov_df, ts_only,
    entity_col=loaded_config["entity_col"],
    date_col=loaded_config["date_col"],
    mode="aggregation",
)

# Filter to only categorical columns for distribution charts
cat_only = [s for s in loaded_config["col_specs"] if s["type"] == "categorical"]
print(f"\nCategorical columns: {[s['name'] for s in cat_only]}")

distribution_chart(
    cov_df, cat_only,
    entity_col=loaded_config["entity_col"],
    date_col=loaded_config["date_col"],
)

## Step 5: KPI Summary

Text-based statistical summary for all column types.

In [ ]:
from earthdaily.agriculture.reporting.viz_engine import kpi_summary

kpi_summary(cov_df, col_specs, entity_col="id", date_col="date")

## Step 6: Column Statistics

Detailed descriptive statistics for individual columns.

In [ ]:
from earthdaily.agriculture.reporting.viz_engine import print_column_stats

print_column_stats(cov_df, "coverage_percent")
print_column_stats(cov_df, "sensor")


## Step 7: Distribution Charts

Histograms for numeric columns, day-of-year distributions for dates,
and value-count bar charts for categoricals.

In [ ]:
from earthdaily.agriculture.reporting.viz_engine import distribution_chart

distribution_chart(cov_df, col_specs, entity_col="id", date_col="date")

## Step 8: Per-Entity Bar Charts

Horizontal bar charts showing each entity's value, sorted and color-coded
by threshold (from `color_scale` in column spec).

In [ ]:
from earthdaily.agriculture.reporting.viz_engine import per_entity_chart

# Aggregate to one row per entity (mean coverage), then show per-entity bars
per_entity_chart(cov_df, col_specs, entity_col="id", date_col="date")

## Step 9: Entity Comparison Chart

Grouped bar chart comparing multiple numeric properties across entities.

In [ ]:
from earthdaily.agriculture.reporting.viz_engine import entity_chart

# Compare coverage_percent and spatial_resolution per entity
entity_chart(
    cov_df,
    col_specs=[s for s in col_specs if s["type"] == "numeric"],
    entity_col="id",
    date_col="date",
)

## Step 10: Time Series Charts

### 10.1 All entities \u2014 individual lines

In [ ]:
from earthdaily.agriculture.reporting.viz_engine import timeseries_chart

# One line per entity
timeseries_chart(
    cov_df, ts_col_specs,
    entity_col="id", date_col="date",
    mode="all",
)

### 10.2 Aggregation mode \u2014 mean +/- std band

In [ ]:
# Central tendency with std deviation band
timeseries_chart(
    cov_df, ts_col_specs,
    entity_col="id", date_col="date",
    mode="aggregation", aggregation="mean",
)

### 10.3 Season overlay \u2014 single entity, year-by-year comparison

In [ ]:
# Pick the entity with the most data points
entity_counts = cov_df["id"].value_counts()
best_entity = entity_counts.index[0]
print(f"Entity with most data: {best_entity} ({entity_counts.iloc[0]} images)")

timeseries_chart(
    cov_df, ts_col_specs,
    entity_col="id", date_col="date",
    mode="season",
    entity_id=best_entity,
    season_start="01/01",
    season_duration=365,
)

## Step 11: Grouped Statistics

Break down a numeric column by a categorical grouping variable.

In [ ]:
from earthdaily.agriculture.reporting.viz_engine import grouped_stats

# Coverage stats broken down by sensor
stats = grouped_stats(cov_df, column="coverage_percent", groupby="sensor")
if stats is not None:
    display(stats)

## Step 12: KPI Group-By Table

Cross-table of KPI statistics broken down by a grouping column.

In [ ]:
from earthdaily.agriculture.reporting.viz_engine import kpi_groupby

# KPI breakdown by sensor
kpi_table = kpi_groupby(
    cov_df, col_specs,
    groupby="sensor",
    entity_col="id", date_col="date",
)
if kpi_table is not None:
    display(kpi_table)

## Step 13: Crosstab Heatmap

Cross-tabulation of two categorical columns as a Plotly heatmap.

In [ ]:
from earthdaily.agriculture.reporting.viz_engine import crosstab_chart

# Sensor vs Mask crosstab
crosstab_chart(cov_df, row_col="sensor", col_col="mask")

In [ ]:
# Normalized (percentage) crosstab
crosstab_chart(cov_df, row_col="sensor", col_col="mask", normalize=True)

## Step 14: Choropleth Map

Geographic visualization of a numeric column on a map. Requires geometry data.

In [ ]:
import geopandas as gpd
from shapely import wkt
from earthdaily.agriculture.reporting.viz_engine import choropleth_map

# Build a GeoDataFrame with mean coverage per entity
entity_summary = cov_df.groupby("id").agg(
    coverage_percent=("coverage_percent", "mean"),
    image_count=("image_id", "count"),
).reset_index()

# Merge geometry from entities
geo_entities = entities[["id", "geometry", "name"]].copy()
geo_entities["geometry"] = geo_entities["geometry"].apply(wkt.loads)
geo_entities = gpd.GeoDataFrame(geo_entities, geometry="geometry", crs="EPSG:4326")

map_df = geo_entities.merge(entity_summary, left_on="id", right_on="id", how="inner")

fig = choropleth_map(
    map_df,
    value_col="coverage_percent",
    entity_col="id",
    name_col="name",
    label="Mean Coverage (%)",
    map_config={"color_ramp": ["#FFEDA0", "#FD8D3C", "#BD0026"]},
)
if fig:
    fig.show()

## Step 15: Scatter Comparison

Compare two numeric columns with a 1:1 line and R²/RMSE statistics.
Useful for comparing results from two different extraction runs or environments.

In [ ]:
from earthdaily.agriculture.reporting.viz_engine import scatter_comparison
import numpy as np

# Simulate a comparison: add noise to coverage_percent as "source 2"
compare_df = cov_df[["id", "date", "coverage_percent"]].copy()
np.random.seed(42)
compare_df["coverage_v2"] = compare_df["coverage_percent"] + np.random.normal(0, 3, len(compare_df))
compare_df["coverage_v2"] = compare_df["coverage_v2"].clip(0, 100)

scatter_comparison(
    compare_df,
    col_1="coverage_percent",
    col_2="coverage_v2",
    label_1="Coverage (Run 1)",
    label_2="Coverage (Run 2)",
    date_col="date",
    title="Coverage Comparison: Run 1 vs Run 2",
)

## Step 16: Time Series Comparison

Side-by-side overlay + scatter for two value columns over time.

In [ ]:
from earthdaily.agriculture.reporting.viz_engine import comparison_chart

# Use the simulated comparison data \u2014 pick entity with most data
entity_compare = compare_df[compare_df["id"] == best_entity].sort_values("date")

comparison_chart(
    entity_compare,
    col_1="coverage_percent",
    col_2="coverage_v2",
    date_col="date",
    label_1="Run 1",
    label_2="Run 2",
    title=f"Coverage Comparison \u2014 Entity {best_entity}",
)

## Summary

| Function | Purpose | Column types |
|---|---|---|
| `generate_viz_config` | Auto-generate col_specs from a DataFrame | all (auto-detected) |
| `load_viz_config` | Load a saved YAML viz config | - |
| `kpi_summary` | Text stats summary | numeric, date, categorical |
| `print_column_stats` | Detailed single-column stats | any |
| `distribution_chart` | Histograms & bar charts | numeric, date, categorical |
| `per_entity_chart` | Per-entity horizontal bars | numeric, date |
| `entity_chart` | Grouped entity comparison | numeric |
| `timeseries_chart` | Line charts (all/agg/season) | timeseries |
| `grouped_stats` | Stats by group | numeric x categorical |
| `kpi_groupby` | KPI cross-table | numeric x categorical |
| `crosstab_chart` | Heatmap cross-tabulation | categorical x categorical |
| `choropleth_map` | Geographic map | numeric + geometry |
| `scatter_comparison` | 1:1 scatter + stats | numeric x numeric |
| `comparison_chart` | Time series overlay + scatter | numeric x numeric + date |